#### Define a few parameters and functions

##### Requirements
This python 3 notebook should only require numpy, pandas, and scipy to do the data analysis

It was originally run with python 3.11.5, numpy 1.24.3, scipy 1.11.3 and pandas 2.1.1

In [ ]:
import numpy as np, pandas as pd, scipy.special as ss, cmath

# set constants: gravitational acceleration (m/s^2), kinematic viscosity (m^2/s), von Karman constant
g, nu, kappa = 9.81, 1e-6, 0.41
a_VB, b_VB = 0.018409368, 0.224072094

# function to calculate the hydraulic radius correction factor
# inputs: alpha = W/H
def calc_FR(alpha): 
    
    # calculate the hydraulic radius correction factor
    return 1/(1 + 2/alpha)


# function to calculate the Vanoni-Brooks correction factor
# inputs: f = gRS/V^2, alpha = W/H, Re = VR/nu
def calc_FVB(f, alpha, Re):
    
    # calculate the ratio of wall friction to total friction (f_wall/f)
    fwall_f = (a_VB / f) * np.real(ss.lambertw(b_VB * (Re/f)**(1/3)))**(-2) 
    
    # calculate the correction factor
    return 1 - fwall_f / (1 + alpha/2)
        
# function to calculate the Guo correction factor
# inputs: f = gRS/V^2, alpha = W/H, ro = 30H/ks = 15H/D50
def calc_FG(f, alpha, ro):
    
    # define important functions
    beta = (1/kappa) * (3/4 - 1/np.pi**2)
    phi = np.exp(-(alpha/np.pi)**(3/2))
    psi = np.sqrt(2/alpha) * (np.log(ro*np.exp(-1))/kappa - (1 + phi)**3/12/kappa + ((1 + np.sqrt(phi))/2)*(1+np.sin(np.pi*phi)/(np.pi*(1+phi))))
    
    # calculate the ratio of wall friction to total friction (tau_wall/tau)
    # two different ways to handle inputs depending on if they are arrays or not
    try: tcomplex = np.array([cmath.sqrt(xi) for xi in f*(1+alpha/2)*(psi**2 + beta**2) - 1])
    except: tcomplex = cmath.sqrt(f*(1+alpha/2)*(psi**2 + beta**2) - 1)
    twt = np.real((psi*tcomplex - beta)**2) / ((psi**2 + beta**2)**2 * f)
        
    # calculate the correction factor
    return 1 - twt/(1+alpha/2)
        
def calculate_important_params(df):
    df = df.copy()
    
    # calculate channel aspect ratio
    df['alpha'] = df.W / df.H

    # calculate hydraulic radius
    df['R'] = df.W * df.H / (df.W + 2*df.H)

    # calculate mean flow velocity 
    df['V'] = df.Q / (df.W * df.H) # flow velocity in m/s

    # calculate Reynolds number
    df['Re'] = df.V * df.R / nu

    # calculate friction factor
    df['f'] = g * df.R * df.S / df.V**2

    # calculate relative roughness
    try:
        df['ks'] = 2 * df.D
        df['ro'] = 30 * df.H / df.ks
    except AttributeError:
        df['ks'] = np.nan # if D is not provided, set ks to NaN
        df['ro'] = np.nan # if D is not provided, set ro to NaN

    # calculate depth-slope product
    df['tau_inf'] = g * df.rho * df.H * df.S

    # calculate the hydraulic radius correction factor and corrected bed stress
    df['FR'] = calc_FR(df.alpha.values)
    df['tau_inf_corr_FR'] = df.FR * df.tau_inf

    return df

def calculate_vanoni_brooks_correction(df):
    df = df.copy()
    df_params = calculate_important_params(df)
    
    # calculate the Vanoni-Brooks correction factor and corrected bed stress
    df['FVB'] = calc_FVB(df_params.f.values, df_params.alpha.values, df_params.Re.values)
    df['tau_corr_VB'] = df.FVB * g * df.rho * df.H * df.S
    
    return df

def calculate_guo_correction(df):
    df = df.copy()
    df_params = calculate_important_params(df)
    
    # calculate the Guo correction factor and corrected bed stress
    df['FG'] = calc_FG(df_params.f.values, df_params.alpha.values, df_params.ro.values)
    df['tau_corr_G'] = df.FG * g * df.rho * df.H * df.S

    return df

#### Enter data

An example dataset is given below that is used to demonstrate how to calculate both the Vanoni Brooks sidewall correction and the Guo sidewall correction. The function can be used on your own data if you create a dataframe with the same columns with the same names.

| column name | parameter | units      |
|-------------|-----------|------------|
| W           | Flume width     | m          |
| H           | Flume height    | m          |
| S           | Flume slope     | m/m        |
| Q           | Discharge | m³/s       |
| rho         | Water density   | kg/m³      |
| rho_s       | Sediment Density | kg/m³ |
| D           | Characteristic grain diameter*     | m          |

*The characteristic grain diameter is not required for the Vanoni-Brooks method

In [ ]:
data = {
    'W': [2.0, 2.0, 2.0, 2.0, 2.0, 0.354, 0.354, 0.354, 0.354, 0.354], # channel width in meters
    'H': [0.387, 0.653, 0.853, 0.908, 1.026, 0.06, 0.136, 0.175, 0.198, 0.101], # channel depth in meters
    'S': [0.0107, 0.0085, 0.0035, 0.0058, 0.004, 0.0227, 0.0112, 0.0047, 0.0032, 0.0131], # channel slope in m/m
    'Q': [1.641, 3.271, 3.271, 4.583, 4.6, 0.0217, 0.0608, 0.0608, 0.0608, 0.0434], # discharge in m^3/s
    'D': [0.02865, 0.02865, 0.02865, 0.02865, 0.02865, 0.00521, 0.00521, 0.00521, 0.00521, 0.00521], # median grain size in meters
    'rho': [995, 995, 995, 995, 995, 995, 995, 995, 995, 995], # water density in kg/m^3
    'rho_s': [2666.6, 2666.6, 2666.6, 2666.6, 2666.6, 2666.6, 2666.6, 2666.6, 2666.6, 2666.6], # sediment density in kg/m^3
    'sample_id': ['example_1', 'example_2', 'example_3', 'example_4', 'example_5',
        'example_6', 'example_7', 'example_8', 'example_9', 'example_10'], # sample identifiers (if needed, can be left out)
    'data_source': ['MPM, 1948', 'MPM, 1948', 'MPM, 1948', 'MPM, 1948', 'MPM, 1948',
        'MPM, 1948', 'MPM, 1948', 'MPM, 1948', 'MPM, 1948', 'MPM, 1948'] # data source identifiers (if needed, can be left out)
}

df = pd.DataFrame(data)
df

#### Calculate bed stress and sidewall corrections

##### Depth-slope product
$$ \tau_\infty = \rho g HS $$

In [ ]:
# calculate depth-slope product
df['tau_inf'] = g * df.rho * df.H * df.S # depth-slope product in pascals
df

#### Calculate corrected bed stress using the Vanoni-Brooks sidewall correction

In [ ]:
# The Vanoni-Brooks correction does not require the median grain size (D) for its calculations, but it is included in the input DataFrame for the Guo correction. 

# calculate the Vanoni-Brooks correction factor and corrected bed stress
df = calculate_vanoni_brooks_correction(df)
df

#### Calculate corrected bed stress using the Guo sidewall correction

In [ ]:
df = calculate_guo_correction(df)
df